# Heston Monte Carlo (QE) — Pricing d'un call européen

Ce notebook montre **pas à pas** comment pricer une option **call européenne** sous le **modèle de Heston** par **Monte Carlo**,
en utilisant le schéma **Quadratic–Exponential (QE) d’Andersen (2008)** pour simuler la variance sans négativité.

## Plan
1. Rappels du modèle et des équations.
2. Implémentation du **QE step** pour la variance.
3. Simulation Monte Carlo avec **antithétiques**.
4. Prix, erreur standard, intervalle de confiance.
5. Comparaison rapide avec **Black–Scholes** (référence).
6. Courbe de **convergence** (prix MC vs nombre de trajectoires).


## 1) Modèle de Heston sous risque-neutre
\begin{align}
dS_t &= (r-q) S_t\,dt + \sqrt{v_t}\, S_t\, dW_t^{(1)},\\
dv_t &= \kappa(\theta - v_t)\,dt + \xi \sqrt{v_t}\, dW_t^{(2)},\qquad \mathrm{corr}(dW^{(1)},dW^{(2)})=\rho.
\end{align}

- $r$: taux sans risque, $q$: dividende continu.
- $v_t$: variance instantanée, $\sigma_t = \sqrt{v_t}$.
- $\kappa$: vitesse de rappel; $\theta$: variance de long terme; $\xi$: vol-of-vol; $\rho$: corrélation.

On simule $v_t$ via le schéma **Quadratic–Exponential (QE)** d'Andersen qui **préserve $v\ge0$** et ajuste les moments.


In [ ]:
# Imports de base
import math
import numpy as np
import matplotlib.pyplot as plt

SQRT_2 = math.sqrt(2.0)
def stdnorm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / SQRT_2))

def bs_call_price(S0, K, T, r, q, sigma):
    if T <= 0:
        return max(S0 - K, 0.0)
    vol = sigma * math.sqrt(T)
    if vol == 0:
        return math.exp(-q*T)*max(S0 - K*math.exp(-(r-q)*T), 0.0)
    d1 = (math.log(S0/K) + (r - q + 0.5*sigma*sigma)*T) / vol
    d2 = d1 - vol
    return S0*math.exp(-q*T)*stdnorm_cdf(d1) - K*math.exp(-r*T)*stdnorm_cdf(d2)


## 2) Schéma QE (Andersen, 2008) pour $v_t$
On met à jour $v_{t+\Delta}$ à partir de $v_t$ en respectant les **moments conditionnels** (moyenne/variance) et
en utilisant deux cas selon $\psi = s^2/m^2$ (détails dans l'article d'Andersen 2008).


In [ ]:
def heston_qe_variance_step(v, dt, kappa, theta, xi, rng):
    exp_kdt = np.exp(-kappa*dt)
    m  = theta + (v - theta) * exp_kdt                      # moyenne conditionnelle
    s2 = (v * xi*xi * exp_kdt / kappa) * (1 - exp_kdt) \
       + (theta * xi*xi / (2.0*kappa)) * (1 - exp_kdt)**2   # variance conditionnelle
    psi = s2 / (m*m + 1e-16)
    psi_c = 1.5  # seuil proposé par Andersen

    z = rng.standard_normal(size=v.shape)
    u = rng.random(size=v.shape)
    v_next = np.empty_like(v)

    mask1 = psi < psi_c
    if np.any(mask1):
        m1, psi1, z1 = m[mask1], psi[mask1], z[mask1]
        b2 = 2.0/psi1 - 1.0 + np.sqrt((2.0/psi1) * (2.0/psi1 - 1.0))
        b  = np.sqrt(b2)
        a  = m1 / (1.0 + b2)
        v_next[mask1] = a * (b + z1)**2

    mask2 = ~mask1
    if np.any(mask2):
        m2, psi2, u2 = m[mask2], psi[mask2], u[mask2]
        p    = (psi2 - 1.0) / (psi2 + 1.0)
        beta = (1.0 - p) / (m2 + 1e-16)
        v2 = np.zeros_like(m2)
        mask_exp = u2 > p
        v2[mask_exp] = np.log((1.0 - p[mask_exp]) / (1.0 - u2[mask_exp])) / (beta[mask_exp] + 1e-16)
        v_next[mask2] = v2
    return v_next, z


## 3) Monte Carlo avec antithétiques
On corrèle le choc du spot et celui de la variance via $\rho$, et on utilise une **moyenne trapézoïdale** pour l'intégrale de variance sur chaque pas.


In [ ]:
def heston_mc_call_qe(
    S0=100.0, K=100.0, T=1.0, r=0.02, q=0.0,
    v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7,
    n_paths=20000, n_steps=252, seed=12345
):
    rng = np.random.default_rng(seed)
    dt  = T / n_steps
    disc = math.exp(-r*T)

    half = n_paths // 2  # antithétiques
    logS0 = math.log(S0)

    prices = []
    for sign in (1.0, -1.0):
        logS = np.full(half, logS0, dtype=np.float64)
        v    = np.full(half, v0,    dtype=np.float64)
        for _ in range(n_steps):
            v_next, z_var = heston_qe_variance_step(v, dt, kappa, theta, xi, rng)
            eps  = rng.standard_normal(size=v.shape)
            z_s  = rho * (sign * z_var) + math.sqrt(max(0.0, 1.0 - rho*rho)) * eps
            v_bar = 0.5 * (v + v_next)
            v_bar = np.maximum(v_bar, 0.0)
            logS += (r - q - 0.5 * v_bar) * dt + np.sqrt(v_bar * dt) * z_s
            v = v_next

        ST = np.exp(logS)
        prices.append(np.maximum(ST - K, 0.0))

    payoffs = np.concatenate(prices)
    price = disc * np.mean(payoffs)
    stderr = disc * np.std(payoffs, ddof=1) / math.sqrt(payoffs.size)
    return price, stderr


## 4) Exemple numérique et IC 95%
On compare aussi à **Black–Scholes** en prenant $\sigma=\sqrt{\theta}$ (référence, pas une vérité).


In [ ]:
# Paramètres
S0, K, T, r, q = 100.0, 100.0, 1.0, 0.02, 0.0
v0, kappa, theta, xi, rho = 0.04, 1.5, 0.04, 0.5, -0.7
n_paths, n_steps = 20000, 252

price_mc, se_mc = heston_mc_call_qe(S0, K, T, r, q, v0, kappa, theta, xi, rho, n_paths, n_steps, seed=12345)
ci_low  = price_mc - 1.96*se_mc
ci_high = price_mc + 1.96*se_mc

sigma_ref = math.sqrt(theta)
price_bs  = bs_call_price(S0, K, T, r, q, sigma_ref)

price_mc, se_mc, (ci_low, ci_high), price_bs

## 5) Convergence: prix MC vs nombre de trajectoires
On fait croître le nombre de trajectoires et on observe la stabilisation du prix.


In [ ]:
grid = [1000, 2000, 4000, 8000, 12000, 20000]
vals = []
for n in grid:
    p, se = heston_mc_call_qe(S0, K, T, r, q, v0, kappa, theta, xi, rho, n_paths=n, n_steps=n_steps, seed=12345)
    vals.append((n, p, se))

ns  = [x[0] for x in vals]
pmc = [x[1] for x in vals]
se  = [x[2] for x in vals]

plt.figure()
plt.plot(ns, pmc, marker='o')
plt.xlabel('Nombre de trajectoires')
plt.ylabel('Prix MC (Heston)')
plt.title('Convergence Monte Carlo (Heston QE)')
plt.grid(True)
plt.show()


## 6) Visualisation de trajectoires simulées
On affiche quelques trajectoires du sous-jacent $S_t$ sous Heston, ainsi que la trajectoire **moyenne**.

In [ ]:
def simulate_paths(
    S0=100.0, T=1.0, r=0.02, q=0.0,
    v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7,
    n_paths=20, n_steps=252, seed=123
):
    rng = np.random.default_rng(seed)
    dt = T/n_steps
    logS = np.full(n_paths, np.log(S0))
    v    = np.full(n_paths, v0)
    paths = np.zeros((n_steps+1, n_paths))
    paths[0] = S0

    for t in range(1, n_steps+1):
        v_next, z_var = heston_qe_variance_step(v, dt, kappa, theta, xi, rng)
        eps = rng.standard_normal(size=v.shape)
        z_s = rho * z_var + math.sqrt(max(0.0, 1.0-rho*rho))*eps
        v_bar = 0.5*(v+v_next)
        v_bar = np.maximum(v_bar, 0.0)
        logS += (r - q - 0.5*v_bar)*dt + np.sqrt(v_bar*dt)*z_s
        v = v_next
        paths[t] = np.exp(logS)
    return paths

# Simuler et tracer quelques paths
n_show = 20
paths = simulate_paths(n_paths=n_show, n_steps=252, T=1.0)
avg_path = paths.mean(axis=1)
time = np.linspace(0, 1.0, paths.shape[0])

plt.figure(figsize=(8,5))
plt.plot(time, paths, lw=0.7, alpha=0.7)
plt.plot(time, avg_path, color='black', lw=2, label='Moyenne')
plt.xlabel('Temps (années)')
plt.ylabel('Sous-jacent $S_t$')
plt.title('Trajectoires simulées sous Heston (QE)')
plt.legend()
plt.grid(True)
plt.show()